# Example: Let's Build a Ternary Commodity Price Tree
In this example, you will build a ternary price tree, i.e., a tree that assumes that the price of a good will go up, stay the same, or decrease tomorrow.

Fill me in.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [3]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Constants
Finally, let's define some constants that we'll use in this example. See the comment next to each constant for what it is, it's permissible values, units, etc.

In [1]:
h = 2; # height of the tree {0,1,h-1} levels
price = 100.0; # initial price of the good
u = 0.04; # up-factor, if it does go up, by how much
d = 0.01; # down-factor, if it goes down, by how much

## Example: Ternary Commodity Price Tree
In this example, we will build a ternary price tree, i.e., a tree that assumes that the price of a good will go up, stay the same, or decrease tomorrow.
To start, we'll setup some constants that we'll use, then we'll create a tree model, and then populate the data in the tree.

In [35]:
array_price_tree = build(ArrayBasedTernaryCommodityPriceTree, (
    h = h, price = price, u = u, d = d)); 

In [45]:
array_price_tree.data

Dict{Int64, Float64} with 13 entries:
  5  => 104.0
  12 => 98.01
  8  => 100.0
  1  => 104.0
  0  => 100.0
  6  => 102.96
  11 => 99.0
  9  => 99.0
  3  => 99.0
  7  => 104.0
  4  => 108.16
  2  => 100.0
  10 => 102.96

### Check: How many nodes do we have in the tree?
If this is a full `k-tree` with `h`-levels, we expect the tree will have:
$$
\begin{equation}
N_{h} = \sum_{j=0}^{h}k^j = \frac{k^{h+1}-1}{k-1}
\end{equation}
$$
nodes where $N_{h}$ includes the final layer of leaves. Let's check this condition with the `array_price_tree` we constructed. The data for the nodes is stored in the `data` field; this should have $N_{h}$ entries.

In [33]:
let
    k = 3; # branching factor
    h = 2; # the height of the tree
    Nₕ = (k^(h+1) - 1)/(k - 1); # expected number of nodes
    @assert Nₕ == length(array_price_tree.data); # if this is NOT true, booom.
end

## Example 2: Building a Ternary Price Tree
Alternatively, we could consider an adjacency-based representation of the price tree that breaks apart the data representation for the connectivity information. 
* We've modeled this case using [the `AdjacencyBasedTernaryCommodityPriceTree` type](src/Types.jl) encoded in the `src/Types.jl` file. We pass some required information [to a `build(...)` method](src/Factory.jl) encoded in the `src/Factory.jl` file to build this type.

In [57]:
adj_price_tree = build(AdjacencyBasedTernaryCommodityPriceTree, (
    h = h, price = price, u = u, d = d));

In [12]:
adj_price_tree.data

Dict{Int64, Float64} with 13 entries:
  5  => 104.0
  12 => 98.01
  8  => 100.0
  1  => 104.0
  0  => 100.0
  6  => 102.96
  11 => 99.0
  9  => 99.0
  3  => 99.0
  7  => 104.0
  4  => 108.16
  2  => 100.0
  10 => 102.96

In [55]:
adj_price_tree.connectivity

Dict{Int64, Vector{Int64}} with 4 entries:
  0 => [1, 2, 3]
  2 => [7, 8, 9]
  3 => [10, 11, 12]
  1 => [4, 5, 6]

### Check: Is the children's numbering correct?
In the adjacency-based representation, we have connectivity information and data, where the connectivity of the tree nodes is stored in the `connectivity::Dict{Int64, Array{Int64,1}}` field of the model. Let's compare this with the numbering we expect in the tree. For a full `k-tree` with `h`-levels, the indices of the children of node $i$, denoted by the set $\mathcal{C}_{i}$, are given by:
$$
\begin{equation}
\mathcal{C}_{i}=\left\{k\cdot{i}+1,k\cdot{i}+2,\dots,k\cdot{i}+k\right\}
\end{equation}
$$
Let's check this condition with the `adj_price_tree` we constructed.

In [66]:
let
    k = 3; # branching factor
    h = 2; # the height of the tree
    i = 2; # parent node index

    # generate expected index set -
    expected_children = Array{Int64,1}(undef, 3);
    for c ∈ 1:k
        expected_children[c] = k*i + c
    end

    # compare
    @assert adj_price_tree.connectivity[i] == expected_children; # if not the same - boom.
end